# Space-map — Section order check

Use SIFT feature matching to compute pairwise similarity between sections,
then:
- visualise the similarity matrix as a heatmap
- suggest the correct section order (greedy nearest-neighbour)
- flag sections that are outliers (potentially damaged or misplaced)

**Data:** set `DATA` to your coordinate CSV. Run all cells — no registration
needed, this works directly from raw coordinates.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

# --- edit this to your local file ---
DATA = 'data/xenium_polyp.csv.gz'
X_COL, Y_COL, LAYER_COL = 'x', 'y', 'layer'
N_LAYERS = None   # limit layers for a quick check; None = all
OUT = 'out/section_order'

## Compute pairwise SIFT similarity and check order

`check_order_from_csv` builds temporary slices, renders density images,
counts SIFT feature matches between every pair, and returns the similarity
matrix plus a suggested section order and any flagged outliers.

In [ ]:
import space_map

os.makedirs(OUT, exist_ok=True)
result = space_map.qc.check_order_from_csv(
    DATA,
    x_col=X_COL, y_col=Y_COL, layer_col=LAYER_COL,
    n_layers=N_LAYERS,
    plot=True,
    save_plot=f'{OUT}/section_similarity.png',
)
print('input order  :', result['ids'])
print('suggested    :', result['suggested_order'])
print('flagged      :', result['flagged'] or '(none)')

## Per-section best-match counts

Each bar is the strongest match that section has to any other — sections
with very low counts may be out of order or damaged.

In [ ]:
import pandas as pd
summary = pd.DataFrame({
    'section': result['ids'],
    'best_match': result['best_match'].astype(int),
    'flagged': [s in result['flagged'] for s in result['ids']],
})
print(summary.to_string(index=False))

## What to do next

- **Order looks wrong?** Re-import using `suggested_order` as the section
  sequence.
- **Flagged sections?** Inspect or drop them before registration:
  ```python
  keep_ids = [s for s in result['ids'] if s not in result['flagged']]
  ```
- **Save the matrix** for further analysis:
  ```python
  import numpy as np
  np.save(f'{OUT}/similarity_matrix.npy', result['matrix'])
  ```